<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/main/%E0%B8%95%E0%B8%A3%E0%B8%A7%E0%B8%88%E0%B8%AA%E0%B8%AD%E0%B8%9A%E0%B8%AA%E0%B8%95%E0%B9%89%E0%B8%AD%E0%B8%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ตรวจสอบสต็อก

In [ ]:
import csv

# 1. สร้าง Class สำหรับเก็บข้อมูลหนังสือ
class Book:
    def __init__(self, book_isbn, book_title, book_author, price, category_id, sub_category_id, shelf_locati, stock_qty):
        self.book_isbn = book_isbn
        self.book_title = book_title
        self.book_author = book_author
        self.price = float(price)
        self.category_id = category_id
        self.sub_category_id = sub_category_id
        self.shelf_locati = shelf_locati
        self.stock_qty = int(stock_qty)

# 2. อ่านไฟล์ CSV และโหลดข้อมูลเข้า book_list
book_list = []

with open("books_data_exact_606.csv", mode="r", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    for row in reader:
        book = Book(
            book_isbn=row["book_isbn"],
            book_title=row["book_title"],
            book_author=row["book_author"],
            price=row["price"],
            category_id=row["category_id"],
            sub_category_id=row.get("sud_category_id", row.get("sub_category_id", "")),
            shelf_locati=row["shelf_locati"],
            stock_qty=row["stock_qty"]
        )
        book_list.append(book)

print(f" {len(book_list)} ")

 606 


เช็คสต็อก

In [ ]:
# ฟังก์ชันเช็กสต็อกขั้นสูง (ค้นได้จาก ISBN, ชื่อเรื่อง, เลขเล่ม, หมวดหลัก, หมวดย่อย)

def check_stock_advanced(keyword):
    search_text = str(keyword).lower().strip()
    found_books = []

    for book in book_list:
        isbn = str(book.book_isbn).lower()
        title = str(book.book_title).lower()
        cat_id = str(book.category_id).lower()
        sub_cat_id = str(book.sub_category_id).lower()

      # แยกค้นหาเลขเล่มโดยเฉพาะ (รองรับทั้งการพิมพ์ "1" หรือ "เล่ม 1")

        is_volume_match = False
        if "เล่ม " in title:
            vol_num = title.split("เล่ม ")[-1].strip()
            if (
                search_text == vol_num
                or search_text == f"เล่ม {vol_num}"
                or search_text == f"เล่ม{vol_num}"
            ):
                is_volume_match = True

        # เงื่อนไขการเช็กทั้งหมด

        if (
            (search_text in isbn)
            or (search_text in title)
            or (search_text == cat_id)
            or (search_text == sub_cat_id)
            or is_volume_match
        ):
            found_books.append(book)

    # แสดงผลการตรวจสอบ

    if not found_books:
        print(f"ไม่พบข้อมูลสต็อกที่ตรงกับคำค้นหา: '{keyword}'")
        return

    print(
        f"ผลการตรวจสอบสต็อกสำหรับ '{keyword}' (พบ {len(found_books)}"
        " รายการ) "
    )
    for b in found_books:
        if b.stock_qty > 0:
            status = f"พร้อมใช้งาน ({b.stock_qty} เล่ม)"
        else:
            status = "สินค้าหมด (0 เล่ม)"

        print(
            f"[{b.book_isbn}] {b.book_title} | หมวดหลัก: {b.category_id} |"
            f" หมวดย่อย: {b.sub_category_id} | สต็อก: {status}"
        )

In [ ]:
#2.ฟังก์ชันยืมหนังสือ
def borrow_book(keyword):
    kw = str(keyword).lower().strip()
    for b in book_list:
        if (kw in b.book_isbn.lower()) or (kw in b.book_title.lower()):
            if b.stock_qty > 0:
                b.stock_qty -= 1
                print(f"ยืมสำเร็จ {b.book_title} (คงเหลือ {b.stock_qty} เล่ม)")
            else:
                print(f"ยืมไม่ได้ หนังสือหมด: {b.book_title}")
            return
    print(f"ไม่พบหนังสือ: {keyword}")


# 3.ฟังก์ชันคืนหนังสือ
def return_book(keyword):
    kw = str(keyword).lower().strip()
    for b in book_list:
        if (kw in b.book_isbn.lower()) or (kw in b.book_title.lower()):
            b.stock_qty += 1
            print(f"คืนสำเร็จ {b.book_title} (สต็อกปัจจุบัน {b.stock_qty} เล่ม)")
            return
    print(f"ไม่พบหนังสือ: {keyword}")

In [ ]:
# 2. เช็กสต็อกด้วยชื่อเรื่อง + เลขเล่ม
check_stock_advanced("ไททัน เล่ม 4")  #ไม่พบข้อมูลเนื่องจากพิมแค่ไททัน เล่ม4 ต้องพิมชื่อเต็ม เช่น ผ่าพิภพไททัน เล่ม 4
check_stock_advanced("Attack on Titan เล่ม 1")

ไม่พบข้อมูลสต็อกที่ตรงกับคำค้นหา: 'ไททัน เล่ม 4'
ไม่พบข้อมูลสต็อกที่ตรงกับคำค้นหา: 'Attack on Titan เล่ม 1'


In [ ]:
# 3. เช็กสต็อกด้วยรหัสหมวดหมู่
check_stock_advanced("05")

ผลการตรวจสอบสต็อกสำหรับ '05' (พบ 201 รายการ) 
[B460101005] ผ่าพิภพไททัน (Attack on Titan) เล่ม 5 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B470101005] ดาบพิฆาตอสูร (Demon Slayer) เล่ม 5 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101005] วันพีช (One Piece) เล่ม 5 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101050] วันพีช (One Piece) เล่ม 50 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101051] วันพีช (One Piece) เล่ม 51 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101052] วันพีช (One Piece) เล่ม 52 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101053] วันพีช (One Piece) เล่ม 53 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101054] วันพีช (One Piece) เล่ม 54 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101055] วันพีช (One Piece) เล่ม 55 | หมวดหลัก: 01 | หมวดย่อย: 01 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B480101056] วันพีช (One Piece) เล่ม

In [ ]:
check_stock_advanced("คุกกี้รัน")

ผลการตรวจสอบสต็อกสำหรับ 'คุกกี้รัน' (พบ 89 รายการ) 
[B560205001] คุกกี้รัน เอาชีวิตรอด เล่ม 1 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205002] คุกกี้รัน เอาชีวิตรอด เล่ม 2 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205003] คุกกี้รัน เอาชีวิตรอด เล่ม 3 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205004] คุกกี้รัน เอาชีวิตรอด เล่ม 4 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205005] คุกกี้รัน เอาชีวิตรอด เล่ม 5 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205006] คุกกี้รัน เอาชีวิตรอด เล่ม 6 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205007] คุกกี้รัน เอาชีวิตรอด เล่ม 7 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205008] คุกกี้รัน เอาชีวิตรอด เล่ม 8 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205009] คุกกี้รัน เอาชีวิตรอด เล่ม 9 | หมวดหลัก: 02 | หมวดย่อย: 05 | สต็อก: พร้อมใช้งาน (5 เล่ม)
[B560205010] คุกกี้รัน เอาชีวิ